# 07 - Gaussian parameter and slot-organisation metrics

This notebook confirms the parameter-space metrics in `Metrics`: per-Gaussian mu / sigma errors masked to active pixels, slot mu statistics, placeholder (inactive-slot) detection precision / recall / F1, the mu-ordering rate, and the permutation-consensus fractions. We construct prediction and GT parameter cubes with a known active / inactive layout and known slot means so the outputs are predictable.

Modules exercised: `pipelines.inference_pipeline.metrics.Metrics` (`_gaussian_param_metrics`, `_slot_mu_stats`, `_placeholder_detection`, `_mu_ordering_rate`, `_permutation_consensus`).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

import matplotlib.pyplot as plt
import numpy             as np
import torch

NP_SEED    = 0
TORCH_SEED = 0
np.random.seed(NP_SEED)
torch.manual_seed(TORCH_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (6.0, 4.0)
plt.rcParams["axes.grid"]   = True
plt.rcParams["grid.alpha"]  = 0.3

print("Repository root :", REPO_ROOT)
print("Torch device    : cpu (CPU only, no checkpoints required)")


## Construct parameter cubes with a known active layout

Two Gaussian slots over an `H x W` grid. Slot 0 is active everywhere with `mu = -3`; slot 1 is active on the left half (`mu = +4`) and a placeholder (amplitude `0`) on the right half. The prediction copies the GT means on active pixels and correctly marks the same placeholders, so detection should be near-perfect.

In [ ]:
from pipelines.inference_pipeline.metrics import Metrics, Result

H, W, K = 8, 8, 2
n_elev  = 12
x_axis  = np.linspace(-10.0, 10.0, n_elev).astype(np.float64)
rng     = np.random.default_rng(0)

params_gt   = np.zeros((3 * K, H, W), dtype=np.float32)
params_pred = np.zeros((3 * K, H, W), dtype=np.float32)

# slot 0: active everywhere, mu = -3
params_gt[0]   = 1.0;  params_gt[1]   = -3.0; params_gt[2]   = 2.0
params_pred[0] = 1.0;  params_pred[1] = -3.0 + 0.2 * rng.standard_normal((H, W)); params_pred[2] = 2.0

# slot 1: active on left half (mu = +4), placeholder on right half (amp = 0)
active1 = np.zeros((H, W), dtype=bool); active1[:, :W // 2] = True
params_gt[3][active1]   = 0.8;  params_gt[4][active1]   = 4.0;  params_gt[5][active1]   = 1.5
params_gt[4][~active1]  = np.nan; params_gt[5][~active1] = np.nan
params_pred[3][active1] = 0.8;  params_pred[4][active1] = 4.0 + 0.2 * rng.standard_normal((H, W))[active1]; params_pred[5][active1] = 1.5
print('slot1 active pixels:', int(active1.sum()), 'of', H * W)


## Wrap in a Result and compute parameter metrics

The curve fields are not exercised here, so we pass smooth placeholders. We call `compute` and pull out the parameter-space keys.

In [ ]:
gt_curves   = np.tile(np.exp(-((x_axis[:, None, None]) ** 2) / 8.0).astype(np.float32), (1, H, W))
pred_curves = gt_curves.copy()

result = Result(
    pred_curves=pred_curves, gt_curves=gt_curves,
    params_pred=params_pred, params_gt=params_gt,
    pixel_mse=np.zeros((H, W), np.float32), pixel_mae=np.zeros((H, W), np.float32),
    pixel_r2=np.ones((H, W), np.float32), pixel_cosine=np.ones((H, W), np.float32),
    pixel_peak_err_idx=np.zeros((H, W), np.int32),
    cube_directory=Path('.'), azimuth_offset=0, range_offset=0,
)
gm = Metrics(result, x_axis, n_gaussians=K).compute(
    elev_indices=np.arange(n_elev), range_indices=np.arange(W), az_indices=np.arange(H))
print('computed', len(gm), 'metric keys')


## Per-slot mu statistics and mu errors

Slot 0 mean mu should be close to `-3` for both pred and GT; slot 1 (active pixels only) close to `+4`. The mu mean-absolute-error per slot should be small (of the order of the `0.2` perturbation).

In [ ]:
for k in range(K):
    print(f'slot {k}: mu_gt_mean={gm[f"slot_{k}_mu_gt_mean"]:+.3f}  '
          f'mu_pred_mean={gm[f"slot_{k}_mu_pred_mean"]:+.3f}  '
          f'mu_mae={gm[f"gauss_{k}_mu_mae"]:.3f}  n_valid={gm[f"gauss_{k}_n_valid"]}')


## Placeholder detection

Because the prediction marks exactly the same slot-1 pixels as inactive, the overall placeholder precision, recall and F1 should all be very close to 1. Slot 0 has no placeholders, so its recall is undefined-but-stable at the floor.

In [ ]:
print('overall precision:', gm['placeholder_precision'])
print('overall recall   :', gm['placeholder_recall'])
print('overall F1       :', gm['placeholder_f1'])
for k in range(K):
    print(f'slot {k}: gt_rate={gm[f"slot_{k}_placeholder_gt_rate"]:.3f}  '
          f'pred_rate={gm[f"slot_{k}_placeholder_pred_rate"]:.3f}  '
          f'f1={gm[f"slot_{k}_placeholder_f1"]:.3f}')


## Ordering and permutation consensus

Slot means are mu = -3 then mu = +4, i.e. strictly increasing, so the mu-ordering rate should be `1.0` on multi-active pixels. With both pred and GT mu-sorted, the identity permutation should dominate, so the consensus fractions should be near `1.0`.

In [ ]:
print('mu_ordering_rate                    :', gm['mu_ordering_rate'])
print('permutation_consensus_dominant_frac :', gm['permutation_consensus_dominant_frac'])
print('permutation_consensus_identity_frac :', gm['permutation_consensus_identity_frac'])


## Visual summary

Bar charts of per-slot mu (GT vs pred), placeholder F1 per slot, and the organisation scalars make the agreement visible at a glance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))
x = np.arange(K)

gt_mu   = [gm[f'slot_{k}_mu_gt_mean']   for k in range(K)]
pred_mu = [gm[f'slot_{k}_mu_pred_mean'] for k in range(K)]
axes[0].bar(x - 0.18, gt_mu,   0.36, label='GT',   color='C0', alpha=0.8)
axes[0].bar(x + 0.18, pred_mu, 0.36, label='Pred', color='C3', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels([f'g{k+1}' for k in range(K)])
axes[0].set_title('mean mu per slot'); axes[0].legend()

f1 = [gm[f'slot_{k}_placeholder_f1'] for k in range(K)] + [gm['placeholder_f1']]
axes[1].bar(range(K + 1), f1, color='C2', alpha=0.8)
axes[1].set_xticks(range(K + 1)); axes[1].set_xticklabels([f'g{k+1}' for k in range(K)] + ['all'])
axes[1].set_ylim(0, 1.08); axes[1].set_title('placeholder F1')

scal = [gm['mu_ordering_rate'], gm['permutation_consensus_dominant_frac'],
        gm['permutation_consensus_identity_frac']]
axes[2].barh(['ordering', 'dominant', 'identity'], scal, color=['C0', 'C2', 'C3'], alpha=0.8)
axes[2].set_xlim(0, 1.08); axes[2].set_title('organisation scalars')

fig.tight_layout()
plt.show()


## Expected visual outcome

Slot means must land near `-3` (g1) and `+4` (g2) for both GT and prediction, mu MAE must be small, placeholder F1 must be near `1.0` overall, and the three organisation scalars (mu-ordering, dominant, identity) must all be close to `1.0`, confirming the constructed slot layout is recovered.